# Rastreabilidade Assistencial no SUS via Data Linkage
### Auditoria da População de Pacientes (jul de 2024 à jun de 2025) entre RNDS e SIA/SIH utilizando CPF/CNS como Identificador Único”

Avaliação da integridade e completude do fluxo de dados assistenciais no SUS, verificando a sobreposição e as exclusões (pacientes) entre os sistemas de Regulação (RNDS) e Faturamento (SIA/SIH) para o ano de 2024. A análise utiliza o CPF ou CNS como chave de linkage, alinhando-se aos princípios da Portaria 6.656/2025.

### Objetivo Principal:
Demonstrar a porcentagem de sobreposição de pacientes e procedimentos entre as bases. Especificamente:
•	Comprovar a rastreabilidade: Determinar a proporção de pacientes concluídos na RNDS (Regulação) que efetivamente aparecem nas bases de faturamento (SIA/SIH).
•	Identificar gargalos/inconsistências: Determinar a proporção de pacientes faturados (SIA/SIH) que não passaram ou não tiveram registro de conclusão na RNDS, evidenciando falhas no registro de Regulação Assistencial.
•	Validar a RNDS: Comprovar estatisticamente que os dados da RNDS, apesar de iniciais, já fornecem uma base válida para estudos de fluxo assistencial e tempo de espera.


### Metodologia de Ciência de Dados e Estatística (Foco em Linkage):
#### ETAPA 1: Analise e tratamento.
Devido ao volume de dados os arquivos estão em Parquet, veja o 'convert_csv_parquet.ipynb' 


In [1]:
# BIBLIOTECA
import pyarrow.parquet as pq
import pyarrow.compute as pc
import pyarrow as pa
import pandas as pd
import os
from collections import Counter
from datetime import datetime

# Contagem de TEMPO de processamento
inicio = datetime.now()
print(f"🔵 Início da execução: {inicio.strftime('%H:%M:%S')}")

🔵 Início da execução: 15:57:48


In [2]:
# Função que resume informações de identificação em um arquivo Parquet
def resumo_identificacao_parquet(parquet_path, batch_size=500_000):
    # Abre o arquivo Parquet para leitura em lotes (batches)
    parquet_file = pq.ParquetFile(parquet_path)

    # Inicializa contadores
    total = sem_cpf = sem_cns = sem_ambos = 0

    # Itera sobre o arquivo em lotes de até 'batch_size' linhas
    # Lendo apenas as colunas "CPF_PAC" e "CNS_PAC"
    for batch in parquet_file.iter_batches(
        batch_size=batch_size,
        columns=["CPF_PAC", "CNS_PAC"]
    ):
        # Cria máscaras booleanas indicando onde os valores estão vazios ("")
        cpf_vazio = pc.equal(batch["CPF_PAC"], "")
        cns_vazio = pc.equal(batch["CNS_PAC"], "")

        # Atualiza o total de linhas processadas
        total += batch.num_rows

        # Conta quantos registros têm CPF vazio
        sem_cpf += pc.sum(cpf_vazio).as_py()

        # Conta quantos registros têm CNS vazio
        sem_cns += pc.sum(cns_vazio).as_py()

        # Conta quantos registros têm CPF e CNS vazios ao mesmo tempo
        sem_ambos += pc.sum(pc.and_(cpf_vazio, cns_vazio)).as_py()

    # Retorna um dicionário com o resumo das contagens
    return {
        "GERAL": total,                  # Total de registros processados
        "SEM CPF": sem_cpf,              # Registros sem CPF
        "SEM CNS": sem_cns,              # Registros sem CNS
        "SEM CPF e SEM CNS": sem_ambos   # Registros sem ambos
    }


In [3]:
# Imprime um cabeçalho para indicar qual base está sendo analisada
print("===== BASE SIH =====")

# Chama a função resumo_identificacao_parquet para gerar o resumo
# do arquivo 'SIH.parquet' localizado na pasta 'base'
resumo_sih = resumo_identificacao_parquet(r"base\SIH.parquet")

# Percorre o dicionário retornado pela função (chaves e valores)
for k, v in resumo_sih.items():
    # Imprime cada chave (descrição) alinhada à esquerda em 25 caracteres
    # e o valor formatado com separador de milhar (ex.: 1,000,000)
    print(f"{k:<25} {v:,}")


===== BASE SIH =====
GERAL                     4,255,843
SEM CPF                   2,144,083
SEM CNS                   2,115,083
SEM CPF e SEM CNS         3,323


In [4]:
# Imprime um cabeçalho para indicar qual base está sendo analisada
print("===== BASE SIA =====")

# Chama a função resumo_identificacao_parquet para gerar o resumo
# do arquivo 'SIA.parquet' localizado na pasta 'base'
resumo_sia = resumo_identificacao_parquet(r"base\SIA.parquet")

# Percorre o dicionário retornado pela função (chaves e valores)
for k, v in resumo_sia.items():
    # Imprime cada chave (descrição) alinhada à esquerda em 25 caracteres
    # e o valor formatado com separador de milhar (ex.: 1,000,000 → 1.000.000)
    print(f"{k:<25} {v:,}")


===== BASE SIA =====
GERAL                     176,260,340
SEM CPF                   171,133,678
SEM CNS                   36,485,231
SEM CPF e SEM CNS         35,622,906


In [5]:
# Função para limpar um arquivo Parquet, removendo registros sem CPF e sem CNS
def limpar_parquet_sem_cpf_e_cns(
    parquet_in: str,          # caminho do arquivo de entrada (.parquet)
    parquet_out: str,         # caminho do arquivo de saída (.parquet)
    compression: str = "zstd",# tipo de compressão usada ao salvar
    batch_size: int = 500_000 # tamanho dos lotes de leitura (para não estourar memória)
):
    # Abre o arquivo Parquet para leitura em lotes
    parquet_file = pq.ParquetFile(parquet_in)
    writer = None  # será usado para escrever o novo arquivo

    try:
        # Itera sobre o arquivo em lotes de até 'batch_size' linhas
        for batch in parquet_file.iter_batches(batch_size=batch_size):
            # Converte o lote em uma tabela PyArrow
            table = pa.Table.from_batches([batch])

            # Cria condição booleana: CPF vazio E CNS vazio
            cond_remover = pc.and_(
                pc.equal(table["CPF_PAC"], ""),
                pc.equal(table["CNS_PAC"], "")
            )

            # Filtra a tabela invertendo a condição (mantém apenas registros válidos)
            table_limpa = pc.filter(table, pc.invert(cond_remover))

            # Se o lote filtrado não tiver linhas, pula para o próximo
            if table_limpa.num_rows == 0:
                continue

            # Inicializa o writer na primeira vez que encontra dados válidos
            if writer is None:
                writer = pq.ParquetWriter(
                    parquet_out,
                    schema=table.schema,   # usa o schema do lote
                    compression=compression,
                    use_dictionary=True,   # otimiza armazenamento
                )

            # Escreve o lote filtrado no arquivo de saída
            writer.write_table(table_limpa)

    finally:
        # Fecha o writer ao final (garante que o arquivo seja salvo corretamente)
        if writer:
            writer.close()


In [6]:
# Mensagem inicial para indicar que a limpeza da base SIH começou
print("🧹 Limpando SIH...")

# Chama a função de limpeza, removendo registros sem CPF e sem CNS
# - parquet_in: arquivo original
# - parquet_out: arquivo limpo que será criado
limpar_parquet_sem_cpf_e_cns(
    parquet_in=r"base\SIH.parquet",
    parquet_out=r"base\SIH_LIMPO.parquet",
)

# Mensagem final confirmando que o arquivo limpo foi gerado
print("✅ SIH limpo criado")


🧹 Limpando SIH...
✅ SIH limpo criado


In [7]:
# Mensagem inicial para indicar que a limpeza da base SIA começou
print("🧹 Limpando SIA...")

# Chama a função de limpeza, removendo registros sem CPF e sem CNS
limpar_parquet_sem_cpf_e_cns(
    parquet_in=r"base\SIA.parquet",
    parquet_out=r"base\SIA_LIMPO.parquet",
)

# Mensagem final confirmando que o arquivo limpo foi gerado
print("✅ SIA limpo criado")


🧹 Limpando SIA...
✅ SIA limpo criado


## BASE DA REGULAÇÃO

In [8]:
# Caminhos dos arquivos de entrada e saída
arquivo_in = r"base\RNDS.parquet"              # Arquivo original
arquivo_out = r"base\RNDS_renomeado.parquet"   # Arquivo que será gerado com colunas renomeadas

# Dicionário de renomeação: mapeia nomes antigos → novos
mapa_colunas = {
    "nu_cpf_paciente": "CPF_PAC",
    "nu_cns_paciente": "CNS_PAC",
    "co_sigtap": "COD_SIGTAP_PROCEDIMENTO",
    "co_cbo": "CBO",
    "sg_uf_estab_executante": "UF_DESC_ATEND",
    "co_municipio_estab_executante": "IBGE_ATEND",
    "co_cnes_estab_executante": "CNES_ATEND",
    "data_solicitacao": "DATA_SOLICITACAO",
    "data_autorizacao": "DATA_AUTORIZACAO",
    "data_execucao": "DATA_EXECUCAO",
    "st_vida_paciente": "ST_VIDA",
    "st_solicitacao": "STATUS",
    "id_registro_sistema_origem": "ID_ORIGEM",
    "ds_sistema_origem": "SIST_ORIGEM"
}

# Abre o arquivo Parquet original para leitura em lotes
parquet_file = pq.ParquetFile(arquivo_in)

# Inicializa o writer (será criado apenas quando necessário)
writer = None

# Processa o arquivo em lotes de até 500 mil linhas
for batch in parquet_file.iter_batches(batch_size=500_000):
    # Converte o lote em uma tabela PyArrow
    table = pa.Table.from_batches([batch])

    # Renomeia as colunas usando o dicionário de mapeamento
    # Se a coluna não estiver no dicionário, mantém o nome original
    table = table.rename_columns([mapa_colunas.get(c, c) for c in table.schema.names])

    # Cria o writer na primeira vez que encontra dados
    if writer is None:
        writer = pq.ParquetWriter(arquivo_out, table.schema)

    # Escreve o lote já com colunas renomeadas no novo arquivo
    writer.write_table(table)

# Fecha o writer ao final para garantir que o arquivo seja salvo corretamente
if writer:
    writer.close()

# Mensagem final confirmando a criação do arquivo renomeado
print("✅ Arquivo renomeado salvo em:", arquivo_out)


✅ Arquivo renomeado salvo em: base\RNDS_renomeado.parquet


In [9]:
# Imprime um cabeçalho para indicar qual base está sendo analisada
print("===== BASE RNDS =====")

# Chama a função resumo_identificacao_parquet para gerar o resumo
# do arquivo 'RNDS_renomeado.parquet' localizado na pasta 'base'
resumo_rnds = resumo_identificacao_parquet(r"base\RNDS_renomeado.parquet")

# Percorre o dicionário retornado pela função (chaves e valores)
for k, v in resumo_rnds.items():
    # Imprime cada chave (descrição) alinhada à esquerda em 25 caracteres
    # e o valor formatado com separador de milhar (ex.: 1,000,000 → 1.000.000)
    print(f"{k:<25} {v:,}")


===== BASE RNDS =====
GERAL                     107,600,060
SEM CPF                   1,541,035
SEM CNS                   0
SEM CPF e SEM CNS         0


In [10]:
# Função para limpar um arquivo Parquet, removendo registros sem CPF
def limpar_parquet_sem_cpf(
    parquet_in: str,          # caminho do arquivo de entrada (.parquet)
    parquet_out: str,         # caminho do arquivo de saída (.parquet)
    compression: str = "zstd",# tipo de compressão usada ao salvar
    batch_size: int = 500_000 # tamanho dos lotes de leitura (para não estourar memória)
):
    # Abre o arquivo Parquet para leitura em lotes
    parquet_file = pq.ParquetFile(parquet_in)
    writer = None  # será usado para escrever o novo arquivo

    try:
        # Itera sobre o arquivo em lotes de até 'batch_size' linhas
        for batch in parquet_file.iter_batches(batch_size=batch_size):
            # Converte o lote em uma tabela PyArrow
            table = pa.Table.from_batches([batch])

            # Cria condição booleana: CPF vazio
            cond_remover = pc.equal(table["CPF_PAC"], "")

            # Filtra a tabela invertendo a condição (mantém apenas registros com CPF preenchido)
            table_limpa = pc.filter(table, pc.invert(cond_remover))

            # Se o lote filtrado não tiver linhas, pula para o próximo
            if table_limpa.num_rows == 0:
                continue

            # Inicializa o writer na primeira vez que encontra dados válidos
            if writer is None:
                writer = pq.ParquetWriter(
                    parquet_out,
                    schema=table.schema,   # usa o schema do lote
                    compression=compression,
                    use_dictionary=True,   # otimiza armazenamento
                )

            # Escreve o lote filtrado no arquivo de saída
            writer.write_table(table_limpa)

    finally:
        # Fecha o writer ao final (garante que o arquivo seja salvo corretamente)
        if writer:
            writer.close()


In [11]:
# Mensagem inicial para indicar que a limpeza da base RNDS começou
print("🧹 Limpando RNDS...")

# Chama a função de limpeza, removendo registros sem CPF
# - parquet_in: arquivo original (já renomeado anteriormente)
# - parquet_out: arquivo limpo que será criado contendo apenas registros com CPF
limpar_parquet_sem_cpf(
    parquet_in=r"base\RNDS_renomeado.parquet",
    parquet_out=r"base\RNDS_com_cpf.parquet",
)

# Mensagem final confirmando que o arquivo limpo foi gerado
print("✅ RNDS limpo criado")


🧹 Limpando RNDS...
✅ RNDS limpo criado


In [12]:
# Nome da coluna que indica status de vida
COL_STATUS = "ST_VIDA"   # ← AjustE: 1 = VIVO, 0 = MORTO

# Função para contar quantos pacientes estão vivos ou mortos em um arquivo Parquet
def contar_vida_parquet(parquet_path, batch_size=500_000):
    # Abre o arquivo Parquet para leitura em lotes
    parquet_file = pq.ParquetFile(parquet_path)
    
    # Inicializa um contador (do módulo collections.Counter)
    contador = Counter()

    # Itera sobre o arquivo em lotes de até 'batch_size' linhas
    for batch in parquet_file.iter_batches(
        batch_size=batch_size,
        columns=[COL_STATUS]   # lê apenas a coluna ST_VIDA
    ):
        # Extrai a coluna ST_VIDA do lote
        arr = batch[COL_STATUS]

        # Remove valores nulos (caso existam)
        arr = pc.drop_null(arr)

        # Converte para lista Python (ex.: [1, 0, 1, 1, 0])
        valores = arr.to_pylist()

        # Atualiza o contador com os valores encontrados
        contador.update(valores)

    # Retorna o contador com a frequência de cada valor
    return contador


In [13]:
# Conta os registros de vida/morte no arquivo RNDS_com_cpf.parquet
contagem = contar_vida_parquet(r"base\RNDS_com_cpf.parquet")

# Obtém os totais com valor padrão 0 caso não exista
total_vivos = contagem.get(1, 0)
total_mortos = contagem.get(0, 0)

# Impressão no formato solicitado
print("===== PACIENTES NA LISTA =====")
print(f"VIVO   {total_vivos:,}")
print(f"MORTO  {total_mortos:,}")


===== PACIENTES NA LISTA =====
VIVO   105,523,722
MORTO  535,303


In [14]:
# Nome da coluna que indica o status da solicitação
COL_STATUS = "STATUS"   # ← Ajuste: define qual coluna será usada

# Função para contar a frequência dos diferentes status em um arquivo Parquet
def contar_status_parquet(parquet_path, batch_size=500_000):
    # Abre o arquivo Parquet para leitura em lotes
    parquet_file = pq.ParquetFile(parquet_path)
    
    # Inicializa um contador (do módulo collections.Counter)
    contador = Counter()

    # Itera sobre o arquivo em lotes de até 'batch_size' linhas
    for batch in parquet_file.iter_batches(
        batch_size=batch_size,
        columns=[COL_STATUS]   # lê apenas a coluna STATUS
    ):
        # Extrai a coluna STATUS do lote
        arr = batch[COL_STATUS]

        # Remove valores nulos (caso existam)
        arr = pc.drop_null(arr)

        # Converte para lista Python (ex.: ["AUTORIZADO", "NEGADO", "PENDENTE"])
        valores = arr.to_pylist()

        # Atualiza o contador com os valores encontrados
        contador.update(valores)

    # Retorna o contador com a frequência de cada status
    return contador


In [15]:
# Conta os registros de status no arquivo RNDS_com_cpf.parquet
contagem = contar_status_parquet(r"base\RNDS_com_cpf.parquet")

print("===== STATUS RNDS =====")

# Percorre os resultados do contador, ordenados do mais frequente para o menos
for status, total in contagem.most_common():
    print(f"{status:<30} {total:,}")


===== STATUS RNDS =====
Atendido/Internado             64,707,871
Agendado                       28,592,746
Falta                          6,180,287
Pendente                       5,859,409
Negado/Cancelado               466,298
Devolvido para o solicitante.  130,649
Excluído                       121,765


In [16]:
# Função para filtrar registros de um arquivo Parquet,
# removendo aqueles que possuem determinados valores na coluna STATUS
def filtrar_status_parquet(
    parquet_in,        # caminho do arquivo de entrada (.parquet)
    parquet_out,       # caminho do arquivo de saída (.parquet)
    status_excluir,    # lista de valores de STATUS que devem ser removidos
    batch_size=500_000 # tamanho dos lotes de leitura (para não estourar memória)
):
    # Abre o arquivo Parquet para leitura em lotes
    parquet_file = pq.ParquetFile(parquet_in)
    writer = None  # será usado para escrever o novo arquivo

    try:
        # Itera sobre o arquivo em lotes de até 'batch_size' linhas
        for batch in parquet_file.iter_batches(batch_size=batch_size):
            # Converte o lote em uma tabela PyArrow
            table = pa.Table.from_batches([batch])

            # Cria condição booleana: STATUS está dentro da lista de exclusão
            cond_excluir = pc.is_in(
                table[COL_STATUS],
                value_set=pa.array(status_excluir)
            )

            # Inverte a condição → mantém apenas registros cujo STATUS não está na lista
            table_ok = pc.filter(table, pc.invert(cond_excluir))

            # Se não houver registros válidos nesse lote, pula para o próximo
            if table_ok.num_rows == 0:
                continue

            # Inicializa o writer na primeira vez que encontra dados válidos
            if writer is None:
                writer = pq.ParquetWriter(
                    parquet_out,
                    schema=table_ok.schema,
                    compression="zstd"   # compressão padrão
                )

            # Escreve o lote filtrado no arquivo de saída
            writer.write_table(table_ok)

    finally:
        # Fecha o writer ao final (garante que o arquivo seja salvo corretamente)
        if writer:
            writer.close()


In [17]:
# Chama a função para filtrar registros do arquivo RNDS_renomeado.parquet
# Exclui todos os registros cujo STATUS esteja em uma das opções da lista
filtrar_status_parquet(
    parquet_in=r"base\RNDS_renomeado.parquet",          # arquivo de entrada
    parquet_out=r"base\RNDS_status_agendados.parquet",  # arquivo de saída filtrado
    status_excluir=[                                    # lista de status a excluir
        "Falta",
        "Pendente",
        "Negado/Cancelado",
        "Devolvido para o solicitante.",
        "Excluído"
    ]
)

# Mensagem final confirmando que o arquivo filtrado foi criado
print("✅ RNDS filtrada por status criada")


✅ RNDS filtrada por status criada


In [18]:
# Excluir arquivos que não serão utilizados... 

# Lista de arquivos que devem ser removidos
lista_arquivos = [
    r"base\RNDS_vivos.parquet",
    r"base\RNDS_renomeado.parquet",
    r"base\RNDS_com_cpf.parquet"
]

# Percorre cada arquivo da lista
for arquivo in lista_arquivos:
    # Verifica se o arquivo existe
    if os.path.exists(arquivo):
        # Remove o arquivo
        os.remove(arquivo)
        print(f"{arquivo} removido.")
    else:
        # Caso não exista, informa ao usuário
        print(f"{arquivo} não encontrado.")


base\RNDS_vivos.parquet não encontrado.
base\RNDS_renomeado.parquet removido.
base\RNDS_com_cpf.parquet removido.


In [19]:
# Função para extrair todos os valores únicos da coluna CNS_PAC em um arquivo Parquet
def extrair_cns_sia(parquet_sia, batch_size=500_000):
    # Abre o arquivo Parquet para leitura em lotes
    parquet_file = pq.ParquetFile(parquet_sia)
    
    # Cria um conjunto vazio (set) para armazenar os valores únicos de CNS
    cns_set = set()

    # Itera sobre o arquivo em lotes de até 'batch_size' linhas
    for batch in parquet_file.iter_batches(
        batch_size=batch_size,
        columns=["CNS_PAC"]   # lê apenas a coluna CNS_PAC
    ):
        # Extrai a coluna CNS_PAC do lote
        col = batch["CNS_PAC"]

        # Remove valores nulos
        col = pc.drop_null(col)

        # Garante que os valores sejam únicos dentro do lote
        col = pc.unique(col)

        # Converte para lista Python e adiciona ao conjunto global
        cns_set.update(col.to_pylist())

    # Retorna o conjunto com todos os CNS únicos encontrados
    return cns_set


In [20]:
# Extrai todos os CNS únicos da base SIH_LIMPO.parquet
cns_sih = extrair_cns_sia(r"base\SIH_LIMPO.parquet")

# Imprime a quantidade total de valores distintos encontrados
print(f"CNS distintos no SIH: {len(cns_sih):,}")

CNS distintos no SIH: 1,876,043


In [21]:
# Extrai todos os CNS únicos da base SIA_LIMPO.parquet
cns_sia = extrair_cns_sia(r"base\SIA_LIMPO.parquet")

# Imprime a quantidade total de valores distintos encontrados
# Usa separador de milhar para facilitar a leitura
print(f"CNS distintos no SIA: {len(cns_sia):,}")


CNS distintos no SIA: 28,218,255


In [22]:
# Célula [dc65a3b8]
# Combina os CNS únicos do SIA e do SIH
cns_combinado = cns_sia.union(cns_sih)

# Calcula a sobreposição (interseção)
cns_sobreposicao = cns_sia.intersection(cns_sih)

# Calcula quantos CNS são exclusivos do SIH (enriquecendo o conjunto do SIA)
cns_exclusivos_sih = cns_sih.difference(cns_sia)

print("===== ANÁLISE DE UNICIDADE E ENRIQUECIMENTO (CNS) =====")
print(f"CNS distintos no SIA: {len(cns_sia):,}")
print(f"CNS distintos no SIH: {len(cns_sih):,}")
print(f"Sobreposição (CNS presentes em SIA e SIH): {len(cns_sobreposicao):,}")
print(f"CNS Exclusivos no SIH (Novos chaves adicionadas): {len(cns_exclusivos_sih):,}")
print(f"Total de CNS Únicos Combinados (SIA + SIH): {len(cns_combinado):,}")
print("=======================================================")

===== ANÁLISE DE UNICIDADE E ENRIQUECIMENTO (CNS) =====
CNS distintos no SIA: 28,218,255
CNS distintos no SIH: 1,876,043
Sobreposição (CNS presentes em SIA e SIH): 866,839
CNS Exclusivos no SIH (Novos chaves adicionadas): 1,009,204
Total de CNS Únicos Combinados (SIA + SIH): 29,227,459


In [26]:
# CÓDIGO FASE 1: CRIAÇÃO DO MAPA ÚNICO CNS -> CPF A PARTIR DA RNDS

import pyarrow.parquet as pq
import pyarrow as pa
import pandas as pd
import os

# Arquivo de Origem (RNDS) e Arquivo de Destino (Mapa)
RNDS_FILE = r"base\RNDS_status_agendados.parquet"
MAP_OUTPUT_FILE = r"base\RNDS_CNS_CPF_MAP.parquet"

BATCH_SIZE = 5_000_000
cns_to_cpf_map = {}

# --- Função de Extração de Dados Otimizada ---
def create_rnds_map_optimized(file_path):
    """Extrai pares CNS_PAC e CPF_PAC da RNDS em blocos e atualiza o mapa global."""
    global cns_to_cpf_map
    
    print(f"Extraindo pares CNS/CPF de: {file_path} (Fonte RNDS)")
    
    if not os.path.exists(file_path):
        print(f"ERRO: Arquivo não encontrado: {file_path}. Verifique o caminho.")
        return

    parquet_file = pq.ParquetFile(file_path)
    
    # Itera em blocos lendo apenas as colunas essenciais
    for i, batch in enumerate(parquet_file.iter_batches(columns=["CNS_PAC", "CPF_PAC"], batch_size=BATCH_SIZE)):
        batch_df = batch.to_pandas()
        
        # Filtra registros onde o CNS ou CPF não é nulo
        # Somente registros com ambas as chaves são úteis para o mapeamento
        clean_df = batch_df.dropna(subset=['CNS_PAC', 'CPF_PAC'])
        
        # Cria um novo mapa local a partir do bloco (CNS como chave, CPF como valor)
        new_mappings = clean_df.set_index('CNS_PAC')['CPF_PAC'].to_dict()
        
        # Atualiza o mapa global. A chave CNS é única, substituindo o último CPF encontrado (o que garante a unicidade)
        cns_to_cpf_map.update(new_mappings)
        
        if (i + 1) % 5 == 0:
            print(f"  {i+1} blocos processados. Mapeamentos únicos no dicionário: {len(cns_to_cpf_map):,}")

# 1. Coleta os dados e constrói o mapa único a partir da RNDS
create_rnds_map_optimized(RNDS_FILE)

print(f"\nTotal de Mapeamentos CNS -> CPF Únicos criados a partir da RNDS: {len(cns_to_cpf_map):,}")

# 2. Converte o dicionário final para DataFrame
unique_map_df = pd.DataFrame(list(cns_to_cpf_map.items()), columns=['CNS_PAC', 'CPF_PAC'])

# 3. Salva o Mapa de Ligação no disco
# Usando pq.write_table para evitar o AttributeError
pq.write_table(pa.Table.from_pandas(unique_map_df), MAP_OUTPUT_FILE)

print(f"✅ Mapa de Mapeamento Único CNS -> CPF (RNDS) criado: {MAP_OUTPUT_FILE}")

# Limpar memória
del cns_to_cpf_map
del unique_map_df

Extraindo pares CNS/CPF de: base\RNDS_status_agendados.parquet (Fonte RNDS)
  5 blocos processados. Mapeamentos únicos no dicionário: 8,577,651
  10 blocos processados. Mapeamentos únicos no dicionário: 11,694,595
  15 blocos processados. Mapeamentos únicos no dicionário: 13,856,124

Total de Mapeamentos CNS -> CPF Únicos criados a partir da RNDS: 15,118,510
✅ Mapa de Mapeamento Único CNS -> CPF (RNDS) criado: base\RNDS_CNS_CPF_MAP.parquet


In [27]:
# Imprime um cabeçalho para indicar qual base está sendo analisada
print("===== BASE MAPA (CPF E CNS UNICOS NA RNDS) =====")

# Chama a função resumo_identificacao_parquet para gerar o resumo
# do arquivo 'SIA.parquet' localizado na pasta 'base'
resumo_sia = resumo_identificacao_parquet(r"base\RNDS_CNS_CPF_MAP.parquet")

# Percorre o dicionário retornado pela função (chaves e valores)
for k, v in resumo_sia.items():
    # Imprime cada chave (descrição) alinhada à esquerda em 25 caracteres
    # e o valor formatado com separador de milhar (ex.: 1,000,000 → 1.000.000)
    print(f"{k:<30} {v:,}")

===== BASE MAPA (CPF E CNS UNICOS NA RNDS) =====
GERAL                          15,118,510
SEM CPF                        286,401
SEM CNS                        0
SEM CPF e SEM CNS              0


In [28]:
# Imprime um cabeçalho para indicar qual base está sendo analisada
print("===== BASE RNDS =====")

# Chama a função resumo_identificacao_parquet para gerar o resumo
# do arquivo 'SIA.parquet' localizado na pasta 'base'
resumo_sia = resumo_identificacao_parquet(r"base\RNDS_status_agendados.parquet")

# Percorre o dicionário retornado pela função (chaves e valores)
for k, v in resumo_sia.items():
    # Imprime cada chave (descrição) alinhada à esquerda em 25 caracteres
    # e o valor formatado com separador de milhar (ex.: 1,000,000 → 1.000.000)
    print(f"{k:<30} {v:,}")

===== BASE RNDS =====
GERAL                          94,606,093
SEM CPF                        1,305,476
SEM CNS                        0
SEM CPF e SEM CNS              0


In [ ]:
+++

In [ ]:
# Função para filtrar registros da RNDS com base nos CNS válidos do SIA
def extrair_rnds_filtrada(
    parquet_rnds,     # arquivo RNDS de entrada (.parquet)
    cns_validos,      # conjunto/lista de CNS válidos (extraídos do SIA)
    parquet_out,      # arquivo de saída (.parquet) com registros filtrados
    batch_size=500_000 # tamanho dos lotes de leitura (para não estourar memória)
):
    # Abre o arquivo RNDS para leitura em lotes
    parquet_file = pq.ParquetFile(parquet_rnds)
    writer = None  # será usado para escrever o novo arquivo

    try:
        # Itera sobre o arquivo em lotes de até 'batch_size' linhas
        for batch in parquet_file.iter_batches(
            batch_size=batch_size,
            # CORREÇÃO: usar os nomes de coluna padronizados
            columns=["CNS_PAC", "CPF_PAC"] 
        ):
            # Converte o lote em uma tabela PyArrow
            table = pa.Table.from_batches([batch])

            # Cria condição: CNS_PAC está dentro da lista de CNS válidos
            cond = pc.is_in(
                table["CNS_PAC"], 
                value_set=pa.array(list(cns_validos))
            )

            # Aplica o filtro → mantém apenas registros com CNS válido
            table_filtrada = pc.filter(table, cond)

            # Se não houver registros nesse lote, pula para o próximo
            if table_filtrada.num_rows == 0:
                continue

            # Inicializa o writer na primeira vez que encontra dados válidos
            if writer is None:
                writer = pq.ParquetWriter(
                    parquet_out,
                    schema=table_filtrada.schema,
                    compression="zstd"   # compressão padrão
                )

            # Escreve o lote filtrado no arquivo de saída
            writer.write_table(table_filtrada)

    finally:
        # Fecha o writer ao final (garante que o arquivo seja salvo corretamente)
        if writer:
            writer.close()


In [ ]:
# 3. Filtra a RNDS usando o conjunto unificado de CNS
extrair_rnds_filtrada(
    parquet_rnds=r"base\RNDS_status_agendados.parquet", # RNDS já limpa e filtrada por status
    cns_validos=cns_combinado,                         # Conjunto UNIFICADO de CNS (SIA + SIH)
    parquet_out=r"base\RNDS_filtrada_pelo_Faturamento_CNS.parquet" # Novo arquivo RNDS filtrado
)
print("✅ RNDS filtrada pelo CNS do Faturamento (SIA/SIH) criada")

In [ ]:
fim = datetime.now()
tempo_total = fim - inicio

horas, resto = divmod(tempo_total.total_seconds(), 3600)
minutos, segundos = divmod(resto, 60)

print(f"✅ Tempo total de execução: {int(horas)}h {int(minutos)}min {int(segundos)}s")